# Uncertainty Quantification: Real-Data Verification (STUDY_PLAN.md Objective 6)

STUDY_PLAN.md's Objective 6 proposed two forms of uncertainty quantification compatible with the single-forward-pass constraint: (a) tightening the existing approximate conformal threshold into a formally correct finite-sample bound, and (b) a closed-form epistemic-uncertainty estimate on the combiner's own fit via a Laplace approximation. Both are now implemented (`router.clopper_pearson_threshold`, `combiner.LogisticRegressionCombiner.epistemic_std`/`.score_mackay_adjusted`). This notebook is the real-data verification pass - does the tightened bound actually hold better than the old one, and does the epistemic-uncertainty estimate behave the way intuition would suggest? One of the two answers turned out to be genuinely surprising, and is investigated rather than smoothed over below.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)
%matplotlib inline

from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.features import featurize
from deployment_reliability.router import clopper_pearson_threshold, conformal_threshold


In [2]:
CACHE_PATH = os.path.join("..", "data", "logit_cache_resnet50.pt")
assert os.path.exists(CACHE_PATH), "run scripts/collect_logits.py resnet50 first"
cache = torch.load(CACHE_PATH)
logits, labels = cache["logits"], cache["labels"]
splits_arr = np.array(cache["splits"])

def mask(name):
    return torch.from_numpy(splits_arr == name)

m_fit, m_cal, m_test, m_a, m_o = [mask(n) for n in ("combiner_fit", "threshold_cal", "id_test", "imagenet_a", "imagenet_o")]
correct = logits.argmax(dim=-1) == labels
phi = featurize(logits)
combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
S = combiner.score(phi)
print("split sizes:", {n: int(mask(n).sum()) for n in ("combiner_fit", "threshold_cal", "id_test", "imagenet_a", "imagenet_o")})
print("PASSED")

split sizes: {'combiner_fit': 1500, 'threshold_cal': 300, 'id_test': 1500, 'imagenet_a': 1935, 'imagenet_o': 2000}
PASSED


## Part 1 — Does the Clopper-Pearson tightening actually matter?

`conformal_threshold`'s own docstring already says its "+1 correction" is "a practical first-pass approximation... not a formally proven bound." Check 1 asks a direct question: on the ACTUAL fixed `threshold_cal`/`id_test` split (no resampling), does the old rule's Execute band ever come out larger than the new one's at the same target risk?

In [3]:
for target_risk in (0.02, 0.05, 0.10):
    tau_approx = conformal_threshold(S[m_cal], correct[m_cal], alpha=target_risk)
    tau_cp = clopper_pearson_threshold(S[m_cal], correct[m_cal], target_risk=target_risk, confidence=0.95)
    n_approx = int((S[m_cal] >= tau_approx).sum())
    n_cp = int((S[m_cal] >= tau_cp).sum())
    # evaluate held-out error on id_test, never used for either calibration
    def held_out_error(tau):
        accepted = S[m_test] >= tau
        if not accepted.any():
            return float("nan"), 0
        return 1.0 - correct[m_test][accepted].float().mean().item(), int(accepted.sum())
    err_approx, n_test_approx = held_out_error(tau_approx)
    err_cp, n_test_cp = held_out_error(tau_cp)
    print(f"target_risk={target_risk:.2f}  approx: cal_n={n_approx:4d} held_out_err={err_approx:.4f} (violates={err_approx>target_risk})   "
          f"cp: cal_n={n_cp:4d} held_out_err={err_cp:.4f} (violates={err_cp>target_risk})")
    assert n_cp <= n_approx, "CP should never accept a larger band than the approximate rule"
print("\nPASSED: Clopper-Pearson never accepts a larger Execute band than the approximate rule")

target_risk=0.02  approx: cal_n= 151 held_out_err=0.0073 (violates=False)   cp: cal_n=   0 held_out_err=0.0000 (violates=False)
target_risk=0.05  approx: cal_n= 194 held_out_err=0.0293 (violates=False)   cp: cal_n= 156 held_out_err=0.0069 (violates=False)
target_risk=0.10  approx: cal_n= 245 held_out_err=0.0969 (violates=False)   cp: cal_n= 208 held_out_err=0.0467 (violates=False)

PASSED: Clopper-Pearson never accepts a larger Execute band than the approximate rule


## Part 2 — Why this validation uses fresh re-splits, not bootstrap resampling

Simulating many repeated calibration/deployment trials means resampling the pooled `threshold_cal`/`id_test` data - but *how* that resampling is done matters as much as how many trials are run. Bootstrap resampling (with replacement) is the more familiar default for this kind of Monte Carlo check, but it **duplicates points**, which breaks the i.i.d. assumption the Clopper-Pearson bound (and split-conformal prediction generally) depends on: a bootstrap resample of size *n* does not carry the statistical information of *n* genuinely independent draws, so the bound's stated `n` overstates its real information content. Run that way, violation rates for the Clopper-Pearson threshold come out far above its nominal target - not because the threshold is wrong, but because the resampling procedure silently violates an assumption the guarantee depends on. The methodologically sound alternative is **fresh, non-overlapping re-splits** of the pooled data, holding the once-fit combiner score `S` fixed (standard split-conformal practice: the score function is fixed, only the calibration/test partition is randomized). Check 2 below uses this protocol, and its results are the ones this notebook and DESIGN.md report.

In [4]:
def violation_rates(target_risk, n_trials=300, seed=0):
    pool_mask = m_cal | m_test
    S_pool, correct_pool = S[pool_mask], correct[pool_mask]
    n_pool = len(S_pool)
    n_cal_size = int(m_cal.sum())
    torch.manual_seed(seed)
    v_approx, v_cp = 0, 0
    for _ in range(n_trials):
        perm = torch.randperm(n_pool)
        cal_idx, test_idx = perm[:n_cal_size], perm[n_cal_size:]
        Sc, Cc = S_pool[cal_idx], correct_pool[cal_idx]
        St, Ct = S_pool[test_idx], correct_pool[test_idx]
        tau_a = conformal_threshold(Sc, Cc, alpha=target_risk)
        tau_cp = clopper_pearson_threshold(Sc, Cc, target_risk=target_risk, confidence=0.95)
        acc_a, acc_cp = St >= tau_a, St >= tau_cp
        if acc_a.any() and (1 - Ct[acc_a].float().mean().item()) > target_risk:
            v_approx += 1
        if acc_cp.any() and (1 - Ct[acc_cp].float().mean().item()) > target_risk:
            v_cp += 1
    return v_approx / n_trials, v_cp / n_trials

for target_risk in (0.05, 0.10):
    rate_approx, rate_cp = violation_rates(target_risk)
    print(f"target_risk={target_risk:.2f}  approx_violation_rate={rate_approx:.3f}  cp_violation_rate={rate_cp:.3f}  (nominal target: <=0.05)")
    assert rate_cp < rate_approx, "CP should violate less often than the approximate rule"
    assert rate_cp < 0.15, "CP violation rate should be roughly near its 5% nominal target, not wildly off"
print("\nPASSED: with a methodologically sound protocol, the CP threshold's violation rate tracks its nominal target;")
print("the approximate rule's does not (34-44% observed across backbones/risk levels - see DESIGN.md for the full table).")

target_risk=0.05  approx_violation_rate=0.420  cp_violation_rate=0.037  (nominal target: <=0.05)


target_risk=0.10  approx_violation_rate=0.427  cp_violation_rate=0.063  (nominal target: <=0.05)

PASSED: with a methodologically sound protocol, the CP threshold's violation rate tracks its nominal target;
the approximate rule's does not (34-44% observed across backbones/risk levels - see DESIGN.md for the full table).


## Part 3 — Does epistemic_std behave correctly? (synthetic sanity check first)

Before trusting anything on real data: does `epistemic_std` actually increase for a point far from the training feature-space extent, in a case where the answer is obvious by construction?

In [5]:
torch.manual_seed(0)
n_synth = 300
phi_synth = torch.randn(n_synth, 3)
y_synth = (torch.rand(n_synth) < torch.sigmoid(phi_synth[:, 0] - phi_synth[:, 1])).float()
synth_combiner = LogisticRegressionCombiner().fit(phi_synth, y_synth)

near_point = torch.zeros(1, 3)
far_point = torch.full((1, 3), 10.0)
std_near = synth_combiner.epistemic_std(near_point).item()
std_far = synth_combiner.epistemic_std(far_point).item()
print(f"epistemic_std at training-data center: {std_near:.4f}")
print(f"epistemic_std far outside training extent: {std_far:.4f}")
assert std_far > 5 * std_near, "a synthetically far point must show much higher epistemic uncertainty"
print("PASSED: the implementation behaves correctly in the textbook case")

epistemic_std at training-data center: 0.1295
epistemic_std far outside training extent: 2.0972
PASSED: the implementation behaves correctly in the textbook case


## Part 4 — A genuinely surprising real-data finding, investigated rather than hidden

Given Part 3 confirms the mechanism works, the natural next question: is `epistemic_std` **higher** on ImageNet-A/O (shifted/OOD data the combiner never saw during fitting) than on `id_test`? Intuition says yes. Check 4 measures it directly.

In [6]:
std_test = combiner.epistemic_std(phi[m_test])
std_a = combiner.epistemic_std(phi[m_a])
std_o = combiner.epistemic_std(phi[m_o])
print(f"mean epistemic_std:  id_test={std_test.mean().item():.5f}  imagenet_a={std_a.mean().item():.5f}  imagenet_o={std_o.mean().item():.5f}")
print(f"ratio imagenet_a/id_test = {std_a.mean().item()/std_test.mean().item():.3f}")
print(f"ratio imagenet_o/id_test = {std_o.mean().item()/std_test.mean().item():.3f}")
print()
print("Contrary to intuition: BOTH ratios are below 1.0 - ImageNet-A/O show LOWER, not higher,")
print("combiner epistemic uncertainty than id_test on this backbone. Investigated below, not ignored.")

mean epistemic_std:  id_test=0.15364  imagenet_a=0.13844  imagenet_o=0.13248
ratio imagenet_a/id_test = 0.901
ratio imagenet_o/id_test = 0.862

Contrary to intuition: BOTH ratios are below 1.0 - ImageNet-A/O show LOWER, not higher,
combiner epistemic uncertainty than id_test on this backbone. Investigated below, not ignored.


### Why: the Laplace covariance is dominated by the unpenalized bias term

`LogisticRegressionCombiner`'s L2 penalty (`combiner.py`) applies only to the feature weights, never the bias - by design, so the bias can freely set the base rate. That means the bias direction is the *least constrained* (highest-variance) direction in the fitted Laplace covariance, and dominates the eigendecomposition.

In [7]:
eigvals, eigvecs = torch.linalg.eigh(combiner._covariance)
print("covariance eigenvalues (ascending):", eigvals.tolist())
print("dominant eigenvector (msp, margin, entropy, energy, l2norm, bias):", eigvecs[:, -1].round(decimals=4).tolist())

dominant_is_mostly_bias = eigvecs[:, -1].abs()[-1].item() > 0.9
assert dominant_is_mostly_bias, "expected the bias coordinate to dominate the largest-variance eigenvector"
print("\nConfirmed: the single largest-uncertainty direction is >90% the bias coordinate.")
print("Since every input's augmented feature vector has the SAME bias entry (a constant 1),")
print("this direction's contribution to epistemic_std is nearly input-independent - what varies")
print("between id_test and imagenet_a is a much smaller, secondary effect from the two features")
print("with the next-largest loadings (energy_score and normalized_entropy), whose specific")
print("fitted-weight combination happens to move imagenet_a's typical values slightly CLOSER to")
print("this combiner's well-determined region, not further from it.")

covariance eigenvalues (ascending): [1.1898055163328536e-05, 0.0014999056002125144, 0.004864935763180256, 0.02731853537261486, 0.03315135836601257, 1.333083987236023]
dominant eigenvector (msp, margin, entropy, energy, l2norm, bias): [0.015699999406933784, -0.0010000000474974513, -0.04340000078082085, -0.12219999730587006, -0.0013000000035390258, 0.9914000034332275]

Confirmed: the single largest-uncertainty direction is >90% the bias coordinate.
Since every input's augmented feature vector has the SAME bias entry (a constant 1),
this direction's contribution to epistemic_std is nearly input-independent - what varies
between id_test and imagenet_a is a much smaller, secondary effect from the two features
with the next-largest loadings (energy_score and normalized_entropy), whose specific
fitted-weight combination happens to move imagenet_a's typical values slightly CLOSER to
this combiner's well-determined region, not further from it.


### A second surprise: epistemic_std is not a correctness predictor either

Given the mechanism above, `epistemic_std` should not be expected to track ordinary predictive (aleatoric) uncertainty any better than it tracks OOD-ness. Checked directly.

In [8]:
from deployment_reliability.router import auroc

a = auroc(-std_test[correct[m_test]], -std_test[~correct[m_test]])
print(f"AUROC(epistemic_std vs correctness on id_test) = {a:.4f}  (0.5 = no relationship)")
print("Below 0.5: higher combiner-epistemic-uncertainty here is (weakly) associated with CORRECT,")
print("not incorrect, predictions - the opposite of naive intuition, consistent with the mechanism")
print("above: epistemic_std measures how well-determined the FITTED WEIGHT VECTOR is given an")
print("input's feature-space DIRECTION, not how likely that input is to be correctly classified.")

AUROC(epistemic_std vs correctness on id_test) = 0.2541  (0.5 = no relationship)
Below 0.5: higher combiner-epistemic-uncertainty here is (weakly) associated with CORRECT,
not incorrect, predictions - the opposite of naive intuition, consistent with the mechanism
above: epistemic_std measures how well-determined the FITTED WEIGHT VECTOR is given an
input's feature-space DIRECTION, not how likely that input is to be correctly classified.


## Summary

**Clopper-Pearson tightening (Objective 6a): confirmed necessary and correct.** On real ResNet-50 data with a methodologically sound fresh-re-split validation protocol, the old approximate conformal threshold violates its stated target risk 42.0-42.7% of the time at the two risk levels tested here (0.420 at target_risk=0.05, 0.427 at target_risk=0.10) - far too often for anything resembling a guarantee, and consistent with the 34-44% range DESIGN.md reports across backbones. The Clopper-Pearson threshold's violation rate (3.7%/6.3%) tracks its nominal target much more closely, and never accepts a larger Execute band than the old rule (Part 1). Part 2 explains why this comparison uses fresh, non-overlapping re-splits rather than bootstrap resampling: the latter breaks the i.i.d. assumption the bound depends on, and would make a correctly-implemented threshold look broken for reasons that have nothing to do with its correctness.

**Laplace epistemic uncertainty (Objective 6b): implemented, verified mathematically correct, but empirically answers a narrower question than "is this input unusual."** `epistemic_std` is checked against a closed-form Fisher-information formula and against `statsmodels`' Wald standard errors (near machine-precision agreement - see `tests/test_combiner.py`), shrinks monotonically with more fitting data, and correctly flags synthetically far-away points (2.10 vs. 0.13, a >15x gap, in Part 3's textbook case). But on real data it does *not* run higher on ImageNet-A/O than on `id_test` (ratios of 0.901 and 0.862, both below 1.0), and is a weak *negative* correctness predictor on `id_test` (AUROC 0.2541, meaning higher epistemic uncertainty leans toward correct, not incorrect, predictions) - both traced to the same mechanism: the fitted covariance's dominant eigenvector is >99% the unpenalized bias coordinate (0.9914 loading), which is identical for every input by construction, so the direction carrying most of the reported "uncertainty" barely varies across inputs at all. This is reported as exactly what it is: a real, investigated, somewhat counter-intuitive property of this specific combiner's epistemic uncertainty, not evidence of a bug - and a useful reminder that `epistemic_std` is a distinct signal from `distribution_shift`/`familiarity` (`ReferenceNormalizer`'s z-scores), not a cheaper substitute for either. A practical consequence: a deployment that wanted "is this input unusual" would need `ReferenceNormalizer` (or `magnitude`/`shift_a`-style signals from notebook 11) rather than `epistemic_std`, since the latter is answering "how well-determined is the fitted weight vector," a question about the *model*, not "how unfamiliar is this *input*."